#### Import the Necessary Libraries

In [ ]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd

import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [ ]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

#### Define Functions for Getting Data From and Setting Data Into Databases

In [ ]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['src_dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dst_dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

#### Declare & Assign Connection Variables for the MySQL Server & Databases with which You'll be Working 

In [ ]:
mysql_args = {
    "uid" : "lawsonpham",
    "pwd" : "Lawson2006",
    "hostname" : "127.0.0.1",  #"wna8fw-mysql.mysql.database.azure.com",
    "src_dbname" : "adventureworks",
    "dst_dbname" : "adventureworks_dw"
}

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args = {
    "user_name" : "lawsonpham",
    "password" : "Lawson2006",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "m6yd7zf",
    "cluster_location" : "atlas", # "local"
    "db_name" : "adventureworks_dw"
}

csv_args = "data/dim_customers.csv"

#### Create the Destination Data Warehouse Database

In [ ]:
conn_str = (f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}"
            f"@{mysql_args['hostname']}")
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()
connection.execute(text(f"DROP DATABASE IF EXISTS `{mysql_args['dst_dbname']}`;"))
connection.execute(text(f"CREATE DATABASE `{mysql_args['dst_dbname']}`;"))
connection.commit()

print(f"Database '{mysql_args['dst_dbname']}' created successfully.")

#### Create the Date Dimension Table
At this point, we have to **execute the script from Lab 2c** that creates and populates a **Date Dimension** table.  Be certain to target this script to the new data warehouse database we just created **(adventureworks_dw)**.

In [ ]:
sql_dim_date = "SELECT date_key, full_date FROM adventureworks_dw.dim_date;"
df_dim_date = get_sql_dataframe(sql_dim_date, **mysql_args)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

#### Extract, Transform & Load

In [ ]:
# Products
sql_products = "SELECT * FROM adventureworks.dim_products_vw;"
df_products = get_sql_dataframe(sql_products, **mysql_args)
df_products.head(2)

In [ ]:
# Clean columns
drop_cols = ['MakeFlag', 'FinishedGoodsFlag', 'Size', 'SizeUnitMeasureCode', 'WeightUnitMeasureCode',
             'Weight', 'DaysToManufacture', 'ProductLine', 'Class', 'Style', 'ProductCategory', 
             'ProductSubcategory', 'ProductModel', 'SellStartDate', 'SellEndDate', 'DiscontinuedDate']
df_products.drop(drop_cols, axis=1, inplace=True)
df_products.head(2)

In [ ]:
# Add to data warehouse database
dataframe = df_products
table_name = 'dim_products'
primary_key = 'product_key'
db_operation = 'insert'

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [ ]:
# Confirm it worked
sql_products = "SELECT * FROM adventureworks_dw.dim_products;"
df_dim_products = get_sql_dataframe(sql_products, **mysql_args)
df_dim_products.head(2)

In [ ]:
# Vendors
sql_vendors = "SELECT * FROM adventureworks.dim_vendors_vw;"
df_vendors = get_sql_dataframe(sql_vendors, **mysql_args)
df_vendors.head(2)

In [ ]:
drop_cols = ['AddressLine2']
df_vendors.drop(drop_cols, axis=1, inplace=True)
df_vendors.head(2)

In [26]:
dataframe = df_vendors
table_name = 'dim_vendors'
primary_key = 'vendor_key'
db_operation = 'insert'

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [34]:
sql_vendors = "SELECT * FROM adventureworks_dw.dim_vendors;"
df_dim_vendors = get_sql_dataframe(sql_vendors, **mysql_args)
df_dim_vendors.head(2)

,vendor_key,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,City,StateProvinceCode,State_Province,PostalCode
0,1,1,INTERNAT0001,International,1,,,Main Office,683 Larch Ct.,Salt Lake City,UT,Utah,84101
1,2,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,,,Main Office,8547 Catherine Way,Tacoma,WA,Washington,98403


In [22]:
# Customers with CSV
df_customers = pd.read_csv(csv_args)
df_customers.head(2)

,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,NaN,Seattle,WA,Washington,0,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,NaN,Renton,WA,Washington,0,98055,US,United States,North America,Northwest


In [24]:
drop_cols = ['AddressLine2', 'IsOnlyStateProvinceFlag']
df_customers.drop(drop_cols, axis=1, inplace=True)
df_customers.head(2)

,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,StateProvinceCode,State_Province,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,WA,Washington,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,WA,Washington,98055,US,United States,North America,Northwest


In [28]:
dataframe = df_customers
table_name = 'dim_customers'
primary_key = 'customer_key'
db_operation = 'insert'

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args) 

In [35]:
sql_customers = "SELECT * FROM adventureworks_dw.dim_customers;"
df_dim_customers = get_sql_dataframe(sql_customers, **mysql_args)
df_dim_customers.head(2)

,customer_key,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,StateProvinceCode,State_Province,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,WA,Washington,98104,US,United States,North America,Northwest
1,2,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,WA,Washington,98055,US,United States,North America,Northwest


In [29]:
client = get_mongo_client(**mongodb_args)

# Gets the path of the Current Working Directory for this Notebook,
# and then Appends the 'data' directory.
data_dir = os.path.join(os.getcwd(), 'data')

json_files = {"employees" : 'dim_employees.json'
             }

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)       

In [30]:
# Employees from MongoDB
client = get_mongo_client(**mongodb_args)

query = {} # Select all elements (columns), and all documents (rows).
collection = "employees"

df_employees = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
df_employees.head(2)

,EmployeeID,NationalIDNumber,LoginID,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,EmailPromotion,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,adventure-works\guy1,16.0,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,0,320-555-0195,1972-05-15 00:00:00,M,M,1996-07-31 00:00:00,0,21,30,1
1,2,253022876,adventure-works\kevin0,6.0,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,2,150-555-0189,1977-06-03 00:00:00,S,M,1997-02-26 00:00:00,0,42,41,1


In [31]:
drop_cols = ['NationalIDNumber', 'LoginID', 'ManagerID', 'EmailPromotion', 'SalariedFlag', 'CurrentFlag']
df_employees.drop(drop_cols, axis=1, inplace=True)
df_employees.head(2)

,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,VacationHours,SickLeaveHours
0,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15 00:00:00,M,M,1996-07-31 00:00:00,21,30
1,2,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03 00:00:00,S,M,1997-02-26 00:00:00,42,41


In [32]:
dataframe = df_employees
table_name = 'dim_employees'
primary_key = 'employee_key'
db_operation = 'insert'

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args) 

In [36]:
sql_employees = "SELECT * FROM adventureworks_dw.dim_employees;"
df_dim_employees = get_sql_dataframe(sql_employees, **mysql_args)
df_dim_employees.head(2)

,employee_key,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,VacationHours,SickLeaveHours
0,1,213,Chris,O,Okelberry,Production Technician - WC60,chris2@adventure-works.com,315-555-0144,1976-09-07 00:00:00,S,M,1999-04-08 00:00:00,16,28
1,2,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15 00:00:00,M,M,1996-07-31 00:00:00,21,30


In [37]:
# Fact Sales Order
sql_fact_sales = "SELECT * FROM adventureworks.fact_sales_orders_vw;"
df_fact_sales = get_sql_dataframe(sql_fact_sales, **mysql_args)
df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardApprovalCode,SubTotal,TaxAmt,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,105041Vi84182,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,4,711,20.1865,80.746
1,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,105041Vi84182,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,2,712,5.1865,10.373


#### Lookup the DateKeys from the Date Dimension Table.

In [38]:
# Order date
df_dim_order_date = df_dim_date.rename(columns={"date_key" : "order_date_key", "full_date" : "OrderDate"})
df_fact_sales.OrderDate = df_fact_sales.OrderDate.astype('datetime64[ns]').dt.date
df_fact_sales = pd.merge(df_fact_sales, df_dim_order_date, on='OrderDate', how='left')
df_fact_sales.drop(['OrderDate'], axis=1, inplace=True)
df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,...,SubTotal,TaxAmt,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal,order_date_key
0,43659,1,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,...,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,4,711,20.1865,80.746,20010701
1,43659,1,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,...,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,2,712,5.1865,10.373,20010701


In [39]:
# Due date
df_dim_due_date = df_dim_date.rename(columns={"date_key" : "order_date_key", "full_date" : "DueDate"})
df_fact_sales.DueDate = df_fact_sales.DueDate.astype('datetime64[ns]').dt.date
df_fact_sales = pd.merge(df_fact_sales, df_dim_due_date, on='DueDate', how='left')
df_fact_sales.drop(['DueDate'], axis=1, inplace=True)
df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,ContactID,...,TaxAmt,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal,order_date_key_x,order_date_key_y
0,43659,1,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,378,...,1971.5149,616.0984,27231.5495,4911-403C-98,4,711,20.1865,80.746,20010701,20010713
1,43659,1,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,378,...,1971.5149,616.0984,27231.5495,4911-403C-98,2,712,5.1865,10.373,20010701,20010713


In [40]:
# Ship date
df_dim_ship_date = df_dim_date.rename(columns={"date_key" : "order_date_key", "full_date" : "ShipDate"})
df_fact_sales.ShipDate = df_fact_sales.ShipDate.astype('datetime64[ns]').dt.date
df_fact_sales = pd.merge(df_fact_sales, df_dim_ship_date, on='ShipDate', how='left')
df_fact_sales.drop(['ShipDate'], axis=1, inplace=True)
df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,ContactID,SalesPersonID,...,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key
0,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,378,279.0,...,616.0984,27231.5495,4911-403C-98,4,711,20.1865,80.746,20010701,20010713,20010708
1,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,676,378,279.0,...,616.0984,27231.5495,4911-403C-98,2,712,5.1865,10.373,20010701,20010713,20010708


#### Next, use Business Keys to lookup corresponding Surrogate Primary Keys in the Dimension tables

In [42]:
# Customers
sql_dim_customers = "SELECT customer_key, CustomerID FROM adventureworks_dw.dim_customers;"
df_dim_customers = get_sql_dataframe(sql_dim_customers, **mysql_args)
df_dim_customers.head(2)

,customer_key,CustomerID
0,1,1
1,2,2


In [43]:
df_fact_sales = pd.merge(df_fact_sales, df_dim_customers, on='CustomerID', how='inner')

df_fact_sales.drop(['CustomerID'], axis=1, inplace=True)

df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,ContactID,SalesPersonID,Sales Territory Group,...,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key,customer_key
0,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,279.0,North America,...,27231.5495,4911-403C-98,4,711,20.1865,80.746,20010701,20010713,20010708,543
1,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,279.0,North America,...,27231.5495,4911-403C-98,2,712,5.1865,10.373,20010701,20010713,20010708,543


In [46]:
# Employees
sql_dim_employees = "SELECT employee_key, EmployeeID FROM adventureworks_dw.dim_employees;"
df_dim_employees = get_sql_dataframe(sql_dim_employees, **mysql_args)
df_dim_employees.head(2)

,employee_key,EmployeeID
0,1,213
1,2,1


In [49]:
df_fact_sales.rename(columns={'SalesPersonID':'EmployeeID'}, inplace=True)
df_fact_sales = pd.merge(df_fact_sales, df_dim_employees, on='EmployeeID', how='inner')

df_fact_sales.drop(['EmployeeID'], axis=1, inplace=True)

df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,ContactID,Sales Territory Group,Sales Territory,...,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key,customer_key,employee_key
0,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,North America,Southeast,...,4911-403C-98,4,711,20.1865,80.746,20010701,20010713,20010708,543,231
1,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,North America,Southeast,...,4911-403C-98,2,712,5.1865,10.373,20010701,20010713,20010708,543,231


In [51]:
# Products
sql_dim_products = "SELECT product_key, ProductID FROM adventureworks_dw.dim_products;"
df_dim_products = get_sql_dataframe(sql_dim_products, **mysql_args)
df_dim_products.head(2)

,product_key,ProductID
0,1,1
1,2,2


In [52]:
df_fact_sales = pd.merge(df_fact_sales, df_dim_products, on='ProductID', how='inner')

df_fact_sales.drop(['ProductID'], axis=1, inplace=True)

df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,ContactID,Sales Territory Group,Sales Territory,...,CarrierTrackingNumber,OrderQty,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key,customer_key,employee_key,product_key
0,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,North America,Southeast,...,4911-403C-98,4,20.1865,80.746,20010701,20010713,20010708,543,231,435
1,43659,1,5,b'\x00',SO43659,PO522145787,10-4020-000676,378,North America,Southeast,...,4911-403C-98,2,5.1865,10.373,20010701,20010713,20010708,543,231,438


In [55]:
drop_cols = ['RevisionNumber', 'OnlineOrderFlag', 'SalesOrderNumber', 'PurchaseOrderNumber',
             'AccountNumber', 'ContactID', 'Sales Territory Group', 'Sales Territory', 'BillToAddressID', 'ShipToAddressID',
             'CreditCardApprovalCode', 'Credit Card Number',
             'Credit Card ExpMonth', 'Credit Card ExpYear', 'CarrierTrackingNumber']
df_fact_sales.drop(drop_cols, axis=1, inplace=True)
df_fact_sales.head(2)

,SalesOrderID,Status,ShipMethod,ShipBase,ShipRate,Credit Card Type,SubTotal,TaxAmt,Freight,TotalDue,OrderQty,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key,customer_key,employee_key,product_key
0,43659,5,CARGO TRANSPORT 5,8.99,1.49,ColonialVoice,24643.9362,1971.5149,616.0984,27231.5495,4,20.1865,80.746,20010701,20010713,20010708,543,231,435
1,43659,5,CARGO TRANSPORT 5,8.99,1.49,ColonialVoice,24643.9362,1971.5149,616.0984,27231.5495,2,5.1865,10.373,20010701,20010713,20010708,543,231,438


In [56]:
dataframe = df_fact_sales
table_name = 'dim_fact_sales_orders'
primary_key = 'fact_sales_order_key'
db_operation = 'insert'

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args) 

In [57]:
sql_fact_sales = "SELECT * FROM adventureworks_dw.dim_fact_sales_orders;"
df_dim_fact_sales = get_sql_dataframe(sql_fact_sales, **mysql_args)
df_dim_fact_sales.head(2)

,fact_sales_order_key,SalesOrderID,Status,ShipMethod,ShipBase,ShipRate,Credit Card Type,SubTotal,TaxAmt,Freight,TotalDue,OrderQty,UnitPrice,LineTotal,order_date_key_x,order_date_key_y,order_date_key,customer_key,employee_key,product_key
0,1,43659,5,CARGO TRANSPORT 5,8.99,1.49,ColonialVoice,24643.9362,1971.5149,616.0984,27231.5495,4,20.1865,80.7460,20010701,20010713,20010708,543,231,435
1,2,43671,5,CARGO TRANSPORT 5,8.99,1.49,Vista,9760.1695,780.8136,244.0042,10784.9873,2,419.4589,838.9178,20010701,20010713,20010708,314,237,111


#### Query — Total Sales Revenue by Customer (fact + dim_customers + dim_date)

In [73]:
sql_total_revenue = """
    SELECT
        c.CustomerID,
        c.City,
        c.`Country_Region`,
        d.calendar_year AS order_year,
        d.month_name AS order_month,
        COUNT(f.SalesOrderID) AS total_orders,
        SUM(f.LineTotal) AS total_revenue
    FROM adventureworks_dw.dim_fact_sales_orders AS f
    INNER JOIN adventureworks_dw.dim_customers AS c
        ON f.customer_key = c.customer_key
    INNER JOIN adventureworks_dw.dim_date AS d
        ON f.order_date_key = d.date_key
    GROUP BY c.CustomerID, c.City, c.`Country_Region`, d.calendar_year, d.month_name
    ORDER BY total_revenue DESC
    LIMIT 10;
"""

In [74]:
df_fact_total_revenue = get_sql_dataframe(sql_total_revenue, **mysql_args)
df_fact_total_revenue

,CustomerID,City,Country_Region,order_year,order_month,total_orders,total_revenue
0,278,Orlando,United States,2001,November,40,252396.672336
1,278,Orlando,United States,2001,August,42,243523.879200
2,623,Minneapolis,United States,2001,August,44,231392.662648
3,623,Minneapolis,United States,2001,November,44,209917.613672
4,623,Minneapolis,United States,2002,May,40,203666.839400
5,4,Austin,United States,2002,October,66,199029.004800
6,4,Austin,United States,2002,July,76,195603.562200
7,278,Orlando,United States,2002,May,40,188654.004400
8,278,Orlando,United States,2002,February,40,179408.544000
9,254,Bellevue,United States,2003,September,122,175567.886272
